In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import scipy.optimize
import matplotlib.pyplot as plt


RISK_FREE_RATE = 0.05  # Risk-free rate for Sharpe ratio calculation

#Tickers
TICKERS = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA', 'META', 'NVDA', 'JPM', 'V', 'JNJ']

#download SPY ETF data for comparison from aug 2025 to aug 2026
#NOTE: end='2026-08-01' is exclusive, so this stops at end-of-July 2026 and excludes August 2026 itself.
#If you want the full Aug 2025-Aug 2026 window, change end to '2026-09-01' here and in the next cell.
SPY = yf.download('SPY', start='2025-08-01', end='2026-08-01')['Close']

[*********************100%***********************]  1 of 1 completed


In [9]:
# Download historical daily adjusted close prices for the specified tickers
prices = {}
daily_returns = {}
for ticker in TICKERS:
    data = yf.download(ticker, start='2025-08-01', end='2026-08-01')['Close']
    prices[ticker] = data
    daily_returns[ticker] = data.pct_change().dropna()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [11]:
returns_df = pd.DataFrame({
    ticker: data.squeeze("columns")
    for ticker, data in daily_returns.items()
})

# SPY daily returns
spy_prices = SPY.squeeze("columns")
spy_returns = spy_prices.pct_change().dropna()
spy_returns = spy_returns.reindex(returns_df.index).dropna()

# return, variance, and volatility per stock + SPY
annual_return = returns_df.mean() * 252
annual_variance = returns_df.var() * 252
annual_volatility = np.sqrt(annual_variance)

stats_df = pd.DataFrame({
    'Annual Return': annual_return,
    'Annual Variance': annual_variance,
    'Annual Volatility': annual_volatility,
})

spy_stats = pd.Series({
    'Annual Return': spy_returns.mean() * 252,
    'Annual Variance': spy_returns.var() * 252,
    'Annual Volatility': np.sqrt(spy_returns.var() * 252),
}, name='SPY')

stats_df = pd.concat([stats_df, spy_stats.to_frame().T])

print("Annualized Return / Variance / Volatility:")
print(stats_df)

Annualized Return / Variance / Volatility:
       Annual Return  Annual Variance  Annual Volatility
AAPL        0.463989         0.066845           0.258543
GOOGL       0.692269         0.103021           0.320968
MSFT       -0.063753         0.100845           0.317562
AMZN        0.294055         0.117510           0.342797
TSLA        0.136194         0.215117           0.463807
META       -0.224120         0.145633           0.381619
NVDA        0.213295         0.132977           0.364660
JPM         0.241210         0.050307           0.224292
V           0.108436         0.048416           0.220036
JNJ         0.472056         0.033785           0.183808
SPY         0.204460         0.016440           0.128217


In [12]:
#creating covariance matrix
returns_cov_matrix = returns_df.cov() * 252  # Annualize

print("Covariance Matrix:")
print(returns_cov_matrix)

Covariance Matrix:
           AAPL     GOOGL      MSFT      AMZN      TSLA      META      NVDA  \
AAPL   0.066845  0.016327  0.009240  0.010530  0.025524  0.017755  0.011837   
GOOGL  0.016327  0.103021  0.012276  0.053102  0.060897  0.039263  0.031960   
MSFT   0.009240  0.012276  0.100845  0.038766  0.032010  0.018007  0.033520   
AMZN   0.010530  0.053102  0.038766  0.117510  0.057230  0.055323  0.037962   
TSLA   0.025524  0.060897  0.032010  0.057230  0.215117  0.062671  0.069430   
META   0.017755  0.039263  0.018007  0.055323  0.062671  0.145633  0.047156   
NVDA   0.011837  0.031960  0.033520  0.037962  0.069430  0.047156  0.132977   
JPM    0.009029  0.014414  0.006788  0.013263  0.017993  0.019089  0.015661   
V      0.012790  0.005014  0.012477  0.013396 -0.001309  0.015318 -0.008275   
JNJ    0.001909 -0.000022 -0.014271 -0.009683 -0.014246 -0.008867 -0.014401   

            JPM         V       JNJ  
AAPL   0.009029  0.012790  0.001909  
GOOGL  0.014414  0.005014 -0.000022

In [13]:
#helper function to calculate portfolio return
def port_return(w, mu):
    return np.dot(w, mu)

#helper function to calculate portfolio volatility
def port_volatility(w, cov_matrix):
    return np.sqrt(np.dot(w.T, np.dot(cov_matrix, w)))



In [14]:
#long-short frontier closed form

#moving inputs into numpy arrays for calculations
mu = annual_return.values
cov_matrix = returns_cov_matrix.values

Sigma_inv = np.linalg.inv(cov_matrix)

#ones vector
ones = np.ones(len(mu))

A = np.dot(ones.T, np.dot(Sigma_inv, ones))
B = np.dot(mu.T, np.dot(Sigma_inv, ones))
C = np.dot(mu.T, np.dot(Sigma_inv, mu))
D = A * C - B ** 2

#return grid; starting near min ad stepping up by 2% till highest stock's return
ret_grid = np.arange(annual_return.min(), annual_return.max(), 0.02)

#loop thru grid and calculate weights, vol, sharpe 
volatilities = []
returns = []
weights = []
for ret in ret_grid:
    # Calculate the weights for the given return (Merton two-fund separation)
    lambda1 = (C - B * ret) / D
    lambda2 = (A * ret - B) / D
    w = Sigma_inv @ (lambda1 * ones + lambda2 * mu)

    # Calculate the volatility for the given weights
    vol = port_volatility(w, cov_matrix)

    # store resulting weights and metrics with a dataframe with columns
    volatilities.append(vol)
    returns.append(ret)
    weights.append(w)

ls_frontier_df = pd.DataFrame({
    'Return': returns,
    'Volatility': volatilities,
    'Weights': weights
})  


## Part 1: Long-only minimum variance frontier + long-short frontier

The long-short (unconstrained) frontier was already built above using the closed-form two-fund solution. For the **long-only** case there's no closed form once we impose $w_i \ge 0$, so we solve a constrained quadratic program at each target return with `scipy.optimize.minimize`.

In [ ]:
#long-only frontier (no closed form once weights are constrained >= 0, so we solve numerically)

def min_var_long_only(target_ret, mu, cov_matrix):
    n = len(mu)
    w0 = np.ones(n) / n  # equal-weight starting guess
    bounds = [(0, 1)] * n  # long-only: no shorting, no leverage on a single name
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},                       # fully invested
        {'type': 'eq', 'fun': lambda w: port_return(w, mu) - target_ret},     # hit target return
    ]
    res = scipy.optimize.minimize(port_volatility, w0, args=(cov_matrix,),
                                   method='SLSQP', bounds=bounds, constraints=constraints,
                                   options={'maxiter': 500, 'ftol': 1e-12})
    return res

lo_returns, lo_vols, lo_weights = [], [], []
for ret in ret_grid:
    res = min_var_long_only(ret, mu, cov_matrix)
    if res.success:
        lo_returns.append(ret)
        lo_vols.append(res.fun)
        lo_weights.append(res.x)

lo_frontier_df = pd.DataFrame({
    'Return': lo_returns,
    'Volatility': lo_vols,
    'Weights': lo_weights
})
print(f"Long-only frontier: {len(lo_returns)} of {len(ret_grid)} target returns were feasible long-only")
lo_frontier_df.head()

In [ ]:
#Q1: plot the long-only and long-short efficient frontiers together
plt.figure(figsize=(9, 6))
plt.plot(ls_frontier_df['Volatility'], ls_frontier_df['Return'], label='Long-Short Frontier', color='tab:blue')
plt.plot(lo_frontier_df['Volatility'], lo_frontier_df['Return'], label='Long-Only Frontier', color='tab:orange')
plt.scatter(annual_volatility[TICKERS], annual_return[TICKERS], color='gray', s=25, label='Individual Stocks')
for t in TICKERS:
    plt.annotate(t, (annual_volatility[t], annual_return[t]), fontsize=8)
plt.xlabel('Volatility (Annualized Std Dev)')
plt.ylabel('Annual Return')
plt.title('Efficient Frontier: Long-Only vs Long-Short (10 Stocks)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Part 2: Add a 5% risk-free bond and repeat

With a risk-free asset available, the long-short case collapses to two-fund separation: every efficient portfolio is a mix of the risk-free asset and one tangency portfolio of stocks, so we can still get a closed form for the stock sub-weights. For long-only + bond we again solve numerically, this time optimizing over the 10 stock weights plus the bond weight (bond weight also constrained to $[0,1]$, no borrowing at the risk-free rate).

In [ ]:
#Q2: efficient frontiers with a 5% risk-free bond added, long-only and long-short

def frontier_long_only_with_bond(ret_grid, mu, cov_matrix, rf):
    n = len(mu)
    rets, vols, wts = [], [], []
    for ret in ret_grid:
        w0 = np.ones(n + 1) / (n + 1)  # last entry is the bond weight
        bounds = [(0, 1)] * (n + 1)

        def port_ret_full(w):
            return np.dot(w[:n], mu) + w[n] * rf

        def port_vol_full(w):
            return port_volatility(w[:n], cov_matrix)  # bond has zero variance/covariance

        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w: port_ret_full(w) - ret},
        ]
        res = scipy.optimize.minimize(port_vol_full, w0, method='SLSQP',
                                       bounds=bounds, constraints=constraints,
                                       options={'maxiter': 500, 'ftol': 1e-12})
        if res.success:
            rets.append(ret)
            vols.append(res.fun)
            wts.append(res.x)
    return rets, vols, wts


def frontier_long_short_with_bond(ret_grid, mu, cov_matrix, rf):
    # two-fund separation: w_stock scales the tangency portfolio, remainder sits in the bond
    excess = mu - rf
    z = Sigma_inv @ excess
    tangency_excess_ret = excess @ z  # scalar used to size the stock allocation
    rets, vols, wts = [], [], []
    for ret in ret_grid:
        w_stock = (ret - rf) / tangency_excess_ret * z
        w_bond = 1 - w_stock.sum()
        vol = port_volatility(w_stock, cov_matrix)
        rets.append(ret)
        vols.append(vol)
        wts.append(np.append(w_stock, w_bond))
    return rets, vols, wts


lo_bond_returns, lo_bond_vols, lo_bond_weights = frontier_long_only_with_bond(ret_grid, mu, cov_matrix, RISK_FREE_RATE)
ls_bond_returns, ls_bond_vols, ls_bond_weights = frontier_long_short_with_bond(ret_grid, mu, cov_matrix, RISK_FREE_RATE)

lo_bond_frontier_df = pd.DataFrame({'Return': lo_bond_returns, 'Volatility': lo_bond_vols, 'Weights': lo_bond_weights})
ls_bond_frontier_df = pd.DataFrame({'Return': ls_bond_returns, 'Volatility': ls_bond_vols, 'Weights': ls_bond_weights})

print(f"Long-only + bond: {len(lo_bond_returns)} of {len(ret_grid)} feasible")
print(f"Long-short + bond: {len(ls_bond_returns)} of {len(ret_grid)} feasible")

In [ ]:
#Q2: plot all four frontiers together (stocks only vs stocks + risk-free bond)
plt.figure(figsize=(9, 6))
plt.plot(ls_frontier_df['Volatility'], ls_frontier_df['Return'], label='Long-Short (stocks only)', color='tab:blue')
plt.plot(lo_frontier_df['Volatility'], lo_frontier_df['Return'], label='Long-Only (stocks only)', color='tab:orange')
plt.plot(ls_bond_frontier_df['Volatility'], ls_bond_frontier_df['Return'], label='Long-Short + Bond', color='tab:green', linestyle='--')
plt.plot(lo_bond_frontier_df['Volatility'], lo_bond_frontier_df['Return'], label='Long-Only + Bond', color='tab:red', linestyle='--')
plt.xlabel('Volatility (Annualized Std Dev)')
plt.ylabel('Annual Return')
plt.title('Efficient Frontiers: With and Without the 5% Risk-Free Bond')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Part 3: Sharpe ratios

$\text{Sharpe} = \dfrac{r_{portfolio} - r_f}{\sigma_{portfolio}}$, computed along each of the four frontiers.

In [ ]:
#Q3: Sharpe ratio for every point on all four frontiers

def sharpe_ratio(ret, vol, rf):
    return (ret - rf) / vol

sharpe_ls = [sharpe_ratio(r, v, RISK_FREE_RATE) for r, v in zip(ls_frontier_df['Return'], ls_frontier_df['Volatility'])]
sharpe_lo = [sharpe_ratio(r, v, RISK_FREE_RATE) for r, v in zip(lo_frontier_df['Return'], lo_frontier_df['Volatility'])]
sharpe_ls_bond = [sharpe_ratio(r, v, RISK_FREE_RATE) for r, v in zip(ls_bond_frontier_df['Return'], ls_bond_frontier_df['Volatility'])]
sharpe_lo_bond = [sharpe_ratio(r, v, RISK_FREE_RATE) for r, v in zip(lo_bond_frontier_df['Return'], lo_bond_frontier_df['Volatility'])]

plt.figure(figsize=(9, 6))
plt.plot(ls_frontier_df['Return'], sharpe_ls, label='Long-Short (stocks only)', color='tab:blue')
plt.plot(lo_frontier_df['Return'], sharpe_lo, label='Long-Only (stocks only)', color='tab:orange')
plt.plot(ls_bond_frontier_df['Return'], sharpe_ls_bond, label='Long-Short + Bond', color='tab:green', linestyle='--')
plt.plot(lo_bond_frontier_df['Return'], sharpe_lo_bond, label='Long-Only + Bond', color='tab:red', linestyle='--')
plt.xlabel('Annual Return')
plt.ylabel('Sharpe Ratio')
plt.title('Sharpe Ratio vs Return, All Four Scenarios')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print('Max Sharpe (Long-Short, stocks only):', max(sharpe_ls), 'at return', ls_frontier_df['Return'][np.argmax(sharpe_ls)])
print('Max Sharpe (Long-Only, stocks only):', max(sharpe_lo), 'at return', lo_frontier_df['Return'][np.argmax(sharpe_lo)])
print('Max Sharpe (Long-Short + Bond):', max(sharpe_ls_bond), 'at return', ls_bond_frontier_df['Return'][np.argmax(sharpe_ls_bond)])
print('Max Sharpe (Long-Only + Bond):', max(sharpe_lo_bond), 'at return', lo_bond_frontier_df['Return'][np.argmax(sharpe_lo_bond)])

**Discussion — building the "best" portfolio from Sharpe ratios:**

Fill in with your actual numbers once you run this on your ticker list, but the logic is: the portfolio that maximizes the Sharpe ratio is the tangency portfolio — the single point on the frontier where a line drawn from the risk-free rate is tangent to the stock-only frontier. Once the risk-free bond is added, the efficient set becomes a straight line (the Capital Market Line) running from the risk-free rate through that tangency portfolio, and every point on it has the *same* Sharpe ratio — you just lever up or down between the bond and the tangency portfolio to hit your desired return.

Two wrinkles show up in the curves above, and both are expected, not bugs:
- **Long-Only + Bond kinks and then declines.** Because short-selling and borrowing at the risk-free rate are both prohibited here (bond weight and every stock weight are bounded to [0, 1]), the Sharpe ratio is flat only up to the return of the *unleveraged* long-only tangency portfolio. Beyond that return, there's no more bond to sell and no borrowing available to lever up further, so the optimizer is forced onto the pure long-only stock frontier, and Sharpe declines from there.
- **Long-Short + Bond dips sharply below the risk-free rate.** For target returns below 5%, the only way to hit that exact return with stocks + a bond (with unlimited shorting allowed) is to short the tangency portfolio and put more than 100% into the bond. That's a real, if economically silly, portfolio — it's the *inefficient* mirror-image branch of the frontier hyperbola, not something any investor would actually choose, since the same volatility is available at a much higher return elsewhere on the frontier. It's worth acknowledging in the write-up but doesn't change which portfolio is "best."

So the "best" portfolio isn't a single return target — it's the tangency portfolio's *mix* of the 10 stocks, and then you dial overall risk up or down by shifting money between that mix and the risk-free bond (up to the leverage limits above) depending on your own risk tolerance, rather than by changing the stock weights themselves.

## Part 4: Betas and the capital market curves

Beta of a stock is $\beta_i = \dfrac{\text{Cov}(r_i, r_{SPY})}{\text{Var}(r_{SPY})}$. Portfolio beta is the weight-weighted average of the individual betas (the risk-free bond has $\beta = 0$).

In [ ]:
#Q4: compute beta for each stock against SPY
spy_variance = spy_returns.var() * 252  # annualize to match cov units, though beta itself is unit-free

betas = {}
for t in TICKERS:
    cov_i_spy = returns_df[t].cov(spy_returns) * 252
    betas[t] = cov_i_spy / spy_variance

betas_series = pd.Series(betas, name='Beta')
print(betas_series)
betas_arr = betas_series[TICKERS].values

In [ ]:
#Q4: portfolio beta = weighted average of individual stock betas (bond weight contributes 0)

def portfolio_beta(w, betas_arr):
    n = len(betas_arr)
    return np.dot(w[:n], betas_arr)

beta_ls = [portfolio_beta(w, betas_arr) for w in ls_frontier_df['Weights']]
beta_lo = [portfolio_beta(w, betas_arr) for w in lo_frontier_df['Weights']]
beta_ls_bond = [portfolio_beta(w, betas_arr) for w in ls_bond_frontier_df['Weights']]
beta_lo_bond = [portfolio_beta(w, betas_arr) for w in lo_bond_frontier_df['Weights']]

#Q4: capital market curves -- return on x-axis, beta on y-axis, for each of the four portfolios
plt.figure(figsize=(9, 6))
plt.plot(ls_frontier_df['Return'], beta_ls, label='Long-Short (stocks only)', color='tab:blue')
plt.plot(lo_frontier_df['Return'], beta_lo, label='Long-Only (stocks only)', color='tab:orange')
plt.plot(ls_bond_frontier_df['Return'], beta_ls_bond, label='Long-Short + Bond', color='tab:green', linestyle='--')
plt.plot(lo_bond_frontier_df['Return'], beta_lo_bond, label='Long-Only + Bond', color='tab:red', linestyle='--')
plt.xlabel('Annual Return')
plt.ylabel('Portfolio Beta')
plt.title('Capital Market Curves: Beta vs Return')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Part 5: Treynor ratios

$\text{Treynor} = \dfrac{r_{portfolio} - r_f}{\beta_{portfolio}}$ -- same idea as Sharpe, but risk is measured with beta (systematic risk) instead of total volatility.

In [ ]:
#Q5: Treynor ratio for every point on all four frontiers

def treynor_ratio(ret, beta, rf):
    return (ret - rf) / beta

treynor_ls = [treynor_ratio(r, b, RISK_FREE_RATE) for r, b in zip(ls_frontier_df['Return'], beta_ls)]
treynor_lo = [treynor_ratio(r, b, RISK_FREE_RATE) for r, b in zip(lo_frontier_df['Return'], beta_lo)]
treynor_ls_bond = [treynor_ratio(r, b, RISK_FREE_RATE) for r, b in zip(ls_bond_frontier_df['Return'], beta_ls_bond)]
treynor_lo_bond = [treynor_ratio(r, b, RISK_FREE_RATE) for r, b in zip(lo_bond_frontier_df['Return'], beta_lo_bond)]

plt.figure(figsize=(9, 6))
plt.plot(ls_frontier_df['Return'], treynor_ls, label='Long-Short (stocks only)', color='tab:blue')
plt.plot(lo_frontier_df['Return'], treynor_lo, label='Long-Only (stocks only)', color='tab:orange')
plt.plot(ls_bond_frontier_df['Return'], treynor_ls_bond, label='Long-Short + Bond', color='tab:green', linestyle='--')
plt.plot(lo_bond_frontier_df['Return'], treynor_lo_bond, label='Long-Only + Bond', color='tab:red', linestyle='--')
plt.xlabel('Annual Return')
plt.ylabel('Treynor Ratio')
plt.title('Treynor Ratio vs Return, All Four Scenarios')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Discussion — building the "best" portfolio from Treynor ratios:**

Sharpe penalizes total volatility (systematic + idiosyncratic risk), while Treynor only penalizes systematic risk (beta) — it implicitly assumes idiosyncratic risk has already been diversified away. Because our 10-stock portfolios still carry meaningful idiosyncratic risk, the return/Sharpe-maximizing mix and the return/Treynor-maximizing mix won't generally agree: Treynor can reward tilting toward low-beta names even if they add total volatility, since that volatility isn't priced by Treynor's denominator. If the two measures point to different "best" portfolios, Sharpe is the more defensible answer here since our 10-stock book isn't diversified enough for Treynor's assumption (that only systematic risk matters) to hold — Treynor is more appropriate for judging a slice of an already well-diversified portfolio, e.g. how a single stock or sector bet fits into a broader index-like allocation.

The same two wrinkles from the Sharpe discussion carry over here: Long-Only + Bond kinks once leverage runs out at the tangency return, and Long-Short + Bond dips at returns below 5% because hitting them requires shorting the tangency portfolio — both are features of the constraints, not errors in the ratio itself.

## Part 6: Optional -- full exercise in Python

Done -- every step above (data download, frontiers, Sharpe, beta, Treynor) is implemented in Python in this notebook rather than Excel.